# Simulación multiactivo con Cholesky
María Fernanda Gómez Flores
Simulación de Procesos Financieros

AAPL, MSFT y NVDA. Capital inicial de $100,000, 10,000 simulaciones y horizonte de 1 año.

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

### 1. Precios y rendimientos
Descargamos los precios del último año y calculamos los rendimientos logarítmicos.

In [ ]:
tickers = ["AAPL", "MSFT", "NVDA"]
prices = yf.download(tickers=tickers, period="1y")["Close"]
prices = prices[tickers].dropna()
rets = np.log(prices / prices.shift(1)).dropna()
rets.head()

### 2. Correlación y covarianza
Calculamos la correlación y la matriz de covarianzas para tomar en cuenta la relación entre las tres acciones.

In [ ]:
R = rets.corr()
Sigma = rets.cov()
print(R)
print(Sigma)

### 3. Cholesky
Descomponemos la matriz de covarianzas para generar números normales correlacionados.

In [ ]:
L = np.linalg.cholesky(Sigma)
L

### 4. Rendimiento y volatilidad
Calculamos el rendimiento promedio y la volatilidad anual con 252 días bursátiles.

In [ ]:
mu = rets.mean() * 252
vol = rets.std() * np.sqrt(252)

print("Rendimiento anual:")
print((mu * 100).round(2))
print("Volatilidad anual:")
print((vol * 100).round(2))

### 5. Simulación
Generamos 10,000 números normales usando la semilla 67 y los transformamos con Cholesky.

In [ ]:
M = 10000
np.random.seed(67)
epsilon = np.random.normal(0, 1, (3, M))
Z = L @ epsilon
Z.T

### 6. Precios futuros
Simulamos los precios de las acciones dentro de un año. La volatilidad ya está incorporada en Z por la matriz de covarianzas.

In [ ]:
T = 1
S0 = prices.iloc[-1].values
ST = S0 * np.exp((mu.values - 0.5 * vol.values**2) * T + np.sqrt(252*T) * Z.T)
ST = pd.DataFrame(ST, columns=tickers)
ST.head()

### 7. Pesos de los portafolios
Calculamos el valor final usando los tres vectores de pesos que vimos en clase.

In [ ]:
V0 = 100000
pesos = {
    "Escenario 1": [1, 0, 0],
    "Escenario 2": [0.33, 0.33, 0.34],
    "Escenario 3": [0.2, 0.2, 0.6]
}

resultados = {}

for nombre, w in pesos.items():
    VT = V0 * np.sum((ST.values / S0) * w, axis=1)
    PnL = VT - V0
    resultados[nombre] = {"VT": VT, "PnL": PnL}

    print(nombre)
    print(f"Valor final promedio: ${np.mean(VT):,.2f}")
    print(f"Ganancia promedio: ${np.mean(PnL):,.2f}")
    print()

### 8. Gráfica del valor final
Comparamos los valores futuros de los tres portafolios antes de restar los $100,000.

In [ ]:
for nombre, datos in resultados.items():
    plt.hist(datos["VT"], bins=50, alpha=0.4, label=nombre)

plt.axvline(V0, color="black", linestyle="--", label="Capital inicial")
plt.xlabel("Valor final ($)")
plt.ylabel("Frecuencia")
plt.title("Valor final de los portafolios")
plt.legend()
plt.show()

### 9. Ganancias y pérdidas
Al valor final le restamos el capital inicial. Los resultados menores que cero son pérdidas.

In [ ]:
for nombre, datos in resultados.items():
    plt.hist(datos["PnL"], bins=50, alpha=0.4, label=nombre)

plt.axvline(0, color="black", linestyle="--")
plt.xlabel("Ganancia o pérdida ($)")
plt.ylabel("Frecuencia")
plt.title("Ganancias y pérdidas")
plt.legend()
plt.show()

### 10. VaR y CVaR
Calculamos el VaR al 95% con el percentil 5 y el CVaR con el promedio de los resultados que quedan debajo de ese percentil.

In [ ]:
for nombre, datos in resultados.items():
    PnL = datos["PnL"]
    q = np.percentile(PnL, 5)
    VaR = -q
    CVaR = -np.mean(PnL[PnL <= q])

    print(nombre)
    print("VaR 95%:", round(VaR, 2))
    print("CVaR 95%:", round(CVaR, 2))
    print()

### 11. Gráficas del VaR y CVaR
Marcamos en cada histograma el percentil 5 y el promedio del 5% de los resultados más bajos.

In [ ]:
for nombre, datos in resultados.items():
    PnL = datos["PnL"]
    q = np.percentile(PnL, 5)
    cvar = np.mean(PnL[PnL <= q])

    plt.hist(PnL, bins=50)
    plt.axvline(q, color="red", linestyle="--", label="VaR 95%")
    plt.axvline(cvar, color="black", linestyle="--", label="CVaR 95%")
    plt.xlabel("Ganancia o pérdida ($)")
    plt.ylabel("Frecuencia")
    plt.title(nombre)
    plt.legend()
    plt.show()